<a href="https://colab.research.google.com/github/Kushagra481/Kernel/blob/main/RetNet_CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# RetNet Implementation in PyTorch for Google Colab
# Based on "Retentive Network: A Successor to Transformer for Large Language Models"

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from typing import Optional, Tuple

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class MultiScaleRetention(nn.Module):
    """
    Multi-Scale Retention mechanism - the core of RetNet
    Combines the benefits of attention and RNN-style sequential processing
    """
    def __init__(self, hidden_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert hidden_dim % num_heads == 0

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)

        # Linear projections for Q, K, V
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # Layer normalization for stability (GroupNorm was causing issues)
        self.norm = nn.LayerNorm(hidden_dim)

        # Retention decay rates (gamma) for each head - learnable parameters
        self.register_parameter('gammas', nn.Parameter(torch.ones(num_heads)))

        # Positional embeddings using complex exponentials
        self.register_buffer('theta', None)

        self.dropout = nn.Dropout(dropout)

    def _get_retention_weights(self, seq_len: int) -> torch.Tensor:
        """Compute retention weights D matrix"""
        # Create causal mask with exponential decay
        positions = torch.arange(seq_len, device=device).float()
        relative_pos = positions.unsqueeze(0) - positions.unsqueeze(1)  # [seq_len, seq_len]

        # Apply causal mask (future positions get 0 weight)
        causal_mask = (relative_pos >= 0).float()

        # Compute retention weights for each head
        retention_weights = torch.zeros(self.num_heads, seq_len, seq_len, device=device)

        for h in range(self.num_heads):
            gamma = torch.sigmoid(self.gammas[h])  # Ensure gamma is in (0,1)
            decay = torch.pow(gamma, relative_pos) * causal_mask
            retention_weights[h] = decay

        return retention_weights

    def _get_theta_matrix(self, seq_len: int) -> torch.Tensor:
        """Compute complex exponential position embeddings"""
        if self.theta is None or self.theta.size(1) < seq_len:
            positions = torch.arange(seq_len, device=device).float()
            # Different frequencies for each head dimension
            freqs = torch.exp(-torch.arange(0, self.head_dim, 2, device=device).float() *
                            (math.log(10000.0) / self.head_dim))

            theta = positions.unsqueeze(-1) * freqs.unsqueeze(0)  # [seq_len, head_dim//2]
            # Create complex exponentials - pad if odd head_dim
            if self.head_dim % 2 == 1:
                theta = F.pad(theta, (0, 1))  # Pad last dimension

            theta_complex = torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)
            theta_complex = theta_complex.flatten(-2)  # [seq_len, head_dim]

            # Expand for all heads
            self.register_buffer('theta', theta_complex.unsqueeze(0).expand(self.num_heads, -1, -1))

        return self.theta[:, :seq_len, :]

    def forward(self, x: torch.Tensor,
                incremental_state: Optional[dict] = None,
                use_parallel: bool = True) -> Tuple[torch.Tensor, Optional[dict]]:
        """
        Forward pass supporting both parallel and recurrent computation

        Args:
            x: Input tensor [batch, seq_len, hidden_dim]
            incremental_state: For recurrent/streaming inference
            use_parallel: Whether to use parallel computation (training) or recurrent (inference)
        """
        batch_size, seq_len, hidden_dim = x.shape

        # Linear projections
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)

        # Rearrange for multi-head attention: [batch, num_heads, seq_len, head_dim]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        if use_parallel:
            # Parallel computation (for training)
            output = self._parallel_retention(Q, K, V)
        else:
            # Recurrent computation (for inference)
            output, incremental_state = self._recurrent_retention(Q, K, V, incremental_state)

        # Reshape back and apply output projection
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, hidden_dim)
        output = self.norm(output)
        output = self.out_proj(output)
        output = self.dropout(output)

        return output, incremental_state

    def _parallel_retention(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
        """Parallel retention computation for training"""
        batch_size, num_heads, seq_len, head_dim = Q.shape

        # Get retention weights
        D = self._get_retention_weights(seq_len)  # [num_heads, seq_len, seq_len]

        # Get positional embeddings
        theta = self._get_theta_matrix(seq_len)  # [num_heads, seq_len, head_dim]

        # Apply positional embeddings to Q and K
        # For simplicity, we'll add them (in practice, complex multiplication is used)
        Q = Q + theta.unsqueeze(0)  # Broadcasting
        K = K + theta.unsqueeze(0)

        # Compute retention: R = (Q @ K^T) ⊙ D
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  # [batch, heads, seq, seq]

        # Apply retention weights (element-wise multiplication)
        retention_scores = scores * D.unsqueeze(0)  # Broadcasting D across batch

        # Apply to values
        output = torch.matmul(retention_scores, V)  # [batch, heads, seq, head_dim]

        return output

    def _recurrent_retention(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
                           incremental_state: Optional[dict]) -> Tuple[torch.Tensor, dict]:
        """Recurrent retention computation for inference"""
        batch_size, num_heads, seq_len, head_dim = Q.shape

        if incremental_state is None or 'prev_key_value' not in incremental_state:
            # Initialize state
            incremental_state = {
                'prev_key_value': torch.zeros(batch_size, num_heads, head_dim, head_dim, device=Q.device),
                'step': 0
            }

        outputs = []
        prev_kv = incremental_state['prev_key_value']
        step = incremental_state['step']

        for t in range(seq_len):
            q_t = Q[:, :, t:t+1, :]  # [batch, heads, 1, head_dim]
            k_t = K[:, :, t, :].unsqueeze(-1)  # [batch, heads, head_dim, 1]
            v_t = V[:, :, t, :].unsqueeze(-2)  # [batch, heads, 1, head_dim]

            # Update key-value state with decay
            gammas_expanded = torch.sigmoid(self.gammas).view(1, -1, 1, 1)
            prev_kv = prev_kv * gammas_expanded + torch.matmul(k_t, v_t)

            # Compute output for current step
            output_t = torch.matmul(q_t, prev_kv) * self.scale
            outputs.append(output_t)

            step += 1

        output = torch.cat(outputs, dim=2)  # [batch, heads, seq_len, head_dim]

        incremental_state['prev_key_value'] = prev_kv
        incremental_state['step'] = step

        return output, incremental_state


class FeedForward(nn.Module):
    """Feed-forward network with SwiGLU activation"""
    def __init__(self, hidden_dim: int, ffn_dim: int, dropout: float = 0.0):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.up_proj = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.down_proj = nn.Linear(ffn_dim, hidden_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # SwiGLU activation: Swish(gate) * up
        gate = F.silu(self.gate_proj(x))  # Swish activation
        up = self.up_proj(x)
        return self.dropout(self.down_proj(gate * up))


class RetNetBlock(nn.Module):
    """Single RetNet block with retention and feed-forward"""
    def __init__(self, hidden_dim: int, num_heads: int, ffn_dim: int, dropout: float = 0.0):
        super().__init__()
        self.retention = MultiScaleRetention(hidden_dim, num_heads, dropout)
        self.feed_forward = FeedForward(hidden_dim, ffn_dim, dropout)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x: torch.Tensor,
                incremental_state: Optional[dict] = None,
                use_parallel: bool = True) -> Tuple[torch.Tensor, Optional[dict]]:
        # Retention with residual connection
        retention_out, incremental_state = self.retention(
            self.norm1(x), incremental_state, use_parallel
        )
        x = x + retention_out

        # Feed-forward with residual connection
        x = x + self.feed_forward(self.norm2(x))

        return x, incremental_state


class RetNet(nn.Module):
    """Complete RetNet model"""
    def __init__(self,
                 vocab_size: int,
                 hidden_dim: int = 512,
                 num_heads: int = 8,
                 num_layers: int = 12,
                 ffn_dim: int = 2048,
                 max_seq_len: int = 2048,
                 dropout: float = 0.1):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.max_seq_len = max_seq_len

        # Token embeddings
        self.embed_tokens = nn.Embedding(vocab_size, hidden_dim)
        self.embed_scale = math.sqrt(hidden_dim)

        # Positional embeddings (learnable)
        self.embed_positions = nn.Embedding(max_seq_len, hidden_dim)

        # RetNet blocks
        self.layers = nn.ModuleList([
            RetNetBlock(hidden_dim, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])

        # Final layer norm and output projection
        self.final_norm = nn.LayerNorm(hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size, bias=False)

        # Tie weights between embedding and output
        self.lm_head.weight = self.embed_tokens.weight

        self.dropout = nn.Dropout(dropout)

    def forward(self,
                input_ids: torch.Tensor,
                incremental_state: Optional[dict] = None,
                use_parallel: bool = True) -> Tuple[torch.Tensor, Optional[dict]]:
        batch_size, seq_len = input_ids.shape

        # Token embeddings
        x = self.embed_tokens(input_ids) * self.embed_scale

        # Positional embeddings
        positions = torch.arange(seq_len, device=input_ids.device)
        x = x + self.embed_positions(positions)

        x = self.dropout(x)

        # Process through RetNet blocks
        if not use_parallel and incremental_state is None:
            incremental_state = [{} for _ in range(self.num_layers)]

        for i, layer in enumerate(self.layers):
            layer_state = incremental_state[i] if incremental_state else None
            x, new_state = layer(x, layer_state, use_parallel)
            if incremental_state is not None:
                incremental_state[i] = new_state

        # Final normalization and projection to vocabulary
        x = self.final_norm(x)
        logits = self.lm_head(x)

        return logits, incremental_state


# Example usage and testing
def test_retnet():
    """Test the RetNet implementation"""
    print("Testing RetNet implementation...")

    # Model configuration
    vocab_size = 50257  # GPT-2 vocab size
    hidden_dim = 512
    num_heads = 8
    num_layers = 6
    seq_len = 128
    batch_size = 2

    # Create model
    model = RetNet(
        vocab_size=vocab_size,
        hidden_dim=hidden_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        max_seq_len=2048
    ).to(device)

    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}")

    # Generate random input
    input_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)

    print(f"Input shape: {input_ids.shape}")

    # Test parallel forward pass (training mode)
    print("\nTesting parallel computation (training)...")
    model.train()
    with torch.amp.autocast('cuda') if device.type == 'cuda' else torch.no_grad():
        logits_parallel, _ = model(input_ids, use_parallel=True)

    print(f"Output shape: {logits_parallel.shape}")
    print(f"Output range: [{logits_parallel.min():.3f}, {logits_parallel.max():.3f}]")

    # Test recurrent forward pass (inference mode)
    print("\nTesting recurrent computation (inference)...")
    model.eval()
    with torch.no_grad():
        logits_recurrent, final_state = model(input_ids, use_parallel=False)

    print(f"Recurrent output shape: {logits_recurrent.shape}")
    print(f"Final state keys: {list(final_state[0].keys()) if final_state else None}")

    # Test incremental generation
    print("\nTesting incremental generation...")
    with torch.no_grad():
        # Start with a shorter sequence
        partial_input = input_ids[:, :seq_len//2]
        logits_partial, state = model(partial_input, use_parallel=False)

        # Continue generation
        next_token = torch.randint(0, vocab_size, (batch_size, 1), device=device)
        logits_next, final_state = model(next_token, incremental_state=state, use_parallel=False)

        print(f"Incremental generation successful!")
        print(f"Next token logits shape: {logits_next.shape}")

    print("\n✅ All tests passed!")

    return model

# Memory usage monitoring
def print_memory_usage():
    if device.type == 'cuda':
        print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
              f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

# Run the test
if __name__ == "__main__":
    print("🚀 Starting RetNet CUDA implementation test in Colab")
    print_memory_usage()

    model = test_retnet()

    print_memory_usage()
    print("\n🎉 RetNet implementation complete and tested!")

    # Simple generation example
    print("\n📝 Running a simple text generation example...")

    model.eval()
    with torch.no_grad():
        # Start with some tokens (this would normally be from a tokenizer)
        prompt = torch.randint(0, 1000, (1, 10), device=device)  # Random "prompt"

        generated = []
        state = None

        # Generate 20 tokens
        current_input = prompt
        for step in range(20):
            logits, state = model(current_input, incremental_state=state, use_parallel=False)

            # Sample next token (simple argmax for demo)
            next_token = logits[:, -1:, :].argmax(dim=-1)  # [1, 1]
            generated.append(next_token.item())

            # Update input for next iteration (only the new token)
            current_input = next_token

        print(f"Generated tokens: {generated[:10]}...")  # Show first 10
        print("✨ Generation complete!")

Using device: cuda
🚀 Starting RetNet CUDA implementation test in Colab
GPU Memory: 0.63 GB allocated, 0.64 GB reserved
Testing RetNet implementation...
Model parameters: 51,965,488
Input shape: torch.Size([2, 128])

Testing parallel computation (training)...
Output shape: torch.Size([2, 128, 50257])
Output range: [-127.188, 526.500]

Testing recurrent computation (inference)...
Recurrent output shape: torch.Size([2, 128, 50257])
Final state keys: ['prev_key_value', 'step']

Testing incremental generation...
Incremental generation successful!
Next token logits shape: torch.Size([2, 1, 50257])

✅ All tests passed!
GPU Memory: 0.84 GB allocated, 1.14 GB reserved

🎉 RetNet implementation complete and tested!

📝 Running a simple text generation example...
Generated tokens: [544, 544, 544, 544, 544, 544, 544, 544, 544, 544]...
✨ Generation complete!
